In [2]:
import json
import pandas as pd
import requests

In [3]:
# Current S&P 500 constituents
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

tables = pd.read_html(
    url,
    storage_options={
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 "
            "Chrome/137.0.0.0 Safari/537.36"
        )
    }
)

sp500 = tables[0]

print(sp500.head())

  Symbol             Security             GICS Sector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry    Headquarters Location  Date added  \
0        Industrial Conglomerates    Saint Paul, Minnesota  1957-03-04   
1               Building Products     Milwaukee, Wisconsin  2017-07-26   
2           Health Care Equipment  North Chicago, Illinois  1957-03-04   
3                   Biotechnology  North Chicago, Illinois  2012-12-31   
4  IT Consulting & Other Services          Dublin, Ireland  2011-07-06   

       CIK      Founded  
0    66740         1902  
1    91142         1916  
2     1800         1888  
3  1551152  2013 (1888)  
4  1467373         1989  


In [4]:
import requests
import pandas as pd

# SEC master ticker list
sec_tickers = requests.get(
    "https://www.sec.gov/files/company_tickers.json",
    headers={
        "User-Agent": "michael_manurung@outlook.com (michael manurung)"
    },
    timeout=30,
).json()

df = pd.DataFrame(sec_tickers.values())
print(df.head())

   cik_str ticker           title
0  1045810   NVDA     NVIDIA CORP
1   320193   AAPL      Apple Inc.
2  1652044  GOOGL   Alphabet Inc.
3   789019   MSFT  MICROSOFT CORP
4  1018724   AMZN  AMAZON COM INC


In [5]:
df.to_csv("sp500_cik_mapping.csv", index=False)

In [6]:
# Merge S&P 500 tickers (Wikipedia) with SEC CIK data to get proper CIK mapping
# sp500 columns: Symbol, CIK, ...
# df columns: cik_str, ticker, title
sp500_cik = sp500[["Symbol"]].merge(
    df, left_on="Symbol", right_on="ticker", how="inner"
)

cik_tickers = sp500_cik["ticker"].values.tolist()

print(f"Matched {len(cik_tickers)} / {len(sp500)} S&P 500 companies")
print(f"Sample: {cik_tickers[:5]}")

# Read and update config.json
with open("config.json", "r") as f:
    config = json.load(f)

config["download_filings"]["cik_tickers"] = cik_tickers

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Wrote {len(cik_tickers)} CIK-ticker pairs to config.json")

Matched 501 / 503 S&P 500 companies
Sample: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']
Wrote 501 CIK-ticker pairs to config.json
